# CMA-ME — Baseline (Local)

**Method:** CMA-ME (Covariance Matrix Adaptation MAP-Elites)  
**Seeds:** 2 (42, 123)  
**Task:** Ant-v5 Unidirectional Gait (4D Gait BD, 10×10×10×10 = 10000 bins)  
**Budget:** N_INIT_SAMPLES + N_EMITTERS × POPSIZE × N_STEPS ≈ 1e5

In [ ]:
# ============================================================
# Cell 0: Install dependencies (run once, then restart kernel)
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cma', 'gymnasium[mujoco]', 'torch', 'numpy', 'pandas', 'matplotlib', '-q'])

0

In [ ]:
# ============================================================
# Cell 1: Setup — load SSLVE modules from local repo
# ============================================================
import sys, os, glob

SSLVE_PATH = os.path.expanduser('~/Downloads/org-Latent-Variable-Evolution 3/dev/SSLVE')
sys.path.insert(0, SSLVE_PATH)
for f in sorted(glob.glob(os.path.join(SSLVE_PATH, '*.py'))):
    exec(open(f).read())

print('Setup complete.')

Setup complete.


In [ ]:
# ============================================================
# Cell 2: Method Hyperparameters
# ============================================================
import numpy as np
import random
import torch

# --- Architecture ---
ARCHITECTURE     = [27, 64, 64, 8]  #@param
OUTPUT_ACTIVATION = 'tanh'  #@param {type:"string"}
MAX_STEPS        = 500   #@param {type:"integer"}
N_EPISODES       = 3     #@param {type:"integer"}
CTRL_COST_WEIGHT = 0.5   #@param {type:"number"}

# --- Gait BD Grid ---
BIN_SIZES        = [10, 10, 10, 10]  #@param

# --- Fitness ---
TOP_K            = 1     #@param {type:"integer"}
MAX_FITNESS      = 500.0 #@param {type:"number"}
GREEDY_MEM       = True  #@param {type:"boolean"}

# --- CMA-ME ---
N_EMITTERS       = 10    #@param {type:"integer"}
SIGMA_INIT       = 0.05  #@param {type:"number"}
POPSIZE          = 50    #@param {type:"integer"}
N_INIT_SAMPLES   = 500   #@param {type:"integer"}
SEPARABLE        = True  #@param {type:"boolean"}

# --- Search budget ---
N_STEPS          = 200   #@param {type:"integer"}

# --- Checkpoint base directory (local) ---
CKPT_BASE_DIR    = os.path.expanduser('~/Downloads/ant_gait_ckpts/Ant_CMAME/')

total_evals = N_INIT_SAMPLES + N_EMITTERS * POPSIZE * N_STEPS
print(f'Method: CMA-ME')
print(f'Architecture: {ARCHITECTURE}, Output: {OUTPUT_ACTIVATION}')
print(f'Gait BD: {BIN_SIZES} ({np.prod(BIN_SIZES)} bins)')
print(f'N_EMITTERS={N_EMITTERS}, POPSIZE={POPSIZE}, SIGMA_INIT={SIGMA_INIT}')
print(f'N_INIT_SAMPLES={N_INIT_SAMPLES}, N_STEPS={N_STEPS}')
print(f'Total evaluations: {total_evals}')
print(f'Checkpoints will be saved to: {CKPT_BASE_DIR}')

Method: CMA-ME
Architecture: [27, 64, 64, 8], Output: tanh
Gait BD: [10, 10, 10, 10] (10000 bins)
N_EMITTERS=10, POPSIZE=50, SIGMA_INIT=0.05
N_INIT_SAMPLES=500, N_STEPS=200
Total evaluations: 100500
Checkpoints will be saved to: /Users/frankh/Downloads/ant_gait_ckpts/Ant_CMAME/


In [ ]:
# ============================================================
# Cell 3: Run All Experiments (2 seeds)
# Each run saves a checkpoint + plots immediately after finishing.
# ============================================================

SEEDS = [123]  # seed 42 already completed

fitness_fn = lambda info: -info['forward_sum']

all_results = []

weight_dim = MLP_Agent(ARCHITECTURE, output_activation=OUTPUT_ACTIVATION).get_weight_dim()
print(f'Weight dim: {weight_dim}')

for seed in SEEDS:
    print(f'\n========== seed={seed} ==========')

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    collector = AntOmniCollector(
        max_steps=MAX_STEPS,
        n_episodes=N_EPISODES,
        ctrl_cost_weight=CTRL_COST_WEIGHT,
        seed=seed,
    )
    bd = AntGaitBD(bin_sizes=BIN_SIZES)
    bm = MAPElitesBM(behavior_descriptor=bd, fitness_fn=fitness_fn, top_k=TOP_K, max_fitness=MAX_FITNESS)

    orchestrator = CMAME(
        agent_class=MLP_Agent,
        architecture=ARCHITECTURE,
        agent_kwargs={'output_activation': OUTPUT_ACTIVATION},
        collector=collector,
        behavior_matching=bm,
        n_emitters=N_EMITTERS,
        sigma_init=SIGMA_INIT,
        popsize=POPSIZE,
        greedy_mem=GREEDY_MEM,
        separable=SEPARABLE,
        n_init_samples=N_INIT_SAMPLES,
    )

    orchestrator.run(n_steps=N_STEPS)

    # Save checkpoint immediately after this run
    ckpt_path = os.path.join(
        CKPT_BASE_DIR,
        f'CMAME_emit{N_EMITTERS}_pop{POPSIZE}_sigma{SIGMA_INIT}_seed{seed}/'
    )
    save_cmame_checkpoint(ckpt_path, orchestrator)
    print(f'Checkpoint saved: {ckpt_path}')

    # Save plots into checkpoint folder
    orchestrator.plot_history(save_path=ckpt_path + 'plot_history.png')
    print(f'Plots saved to: {ckpt_path}')

    # Print results
    f_min, f_mean, f_max = bm.fitness_stats()
    qd  = bm.qd_score()
    cov = bm.coverage()
    print(f'QD-score: {qd:.4f} | Coverage: {cov:.4f} | Best fitness: {f_min:.6f}')
    print(f'Total evaluations: {orchestrator.total_evals}')

    all_results.append({
        'seed': seed,
        'qd_score': qd, 'coverage': cov,
    })

print('\n====== All runs complete ======')

Weight dim: 6472

========== seed=123 ==========

--- CMAME Step 1/200 ---
Init: 500/500 [74s elapsed, 0s remaining]]
Archive: 117, Bins: 117, Coverage: 0.0117, Fitness min/mean/max: -98.76/-3.82/45.78, QD-score: 58946.8759, Evals: 500

--- CMAME Step 2/200 ---
Archive: 243, Bins: 243, Coverage: 0.0243, Fitness min/mean/max: -98.76/-5.48/45.78, QD-score: 122831.8904, Evals: 1000

--- CMAME Step 3/200 ---
Archive: 340, Bins: 340, Coverage: 0.0340, Fitness min/mean/max: -98.76/-5.84/64.61, QD-score: 171984.1463, Evals: 1500

--- CMAME Step 4/200 ---
Archive: 412, Bins: 412, Coverage: 0.0412, Fitness min/mean/max: -98.76/-7.88/43.99, QD-score: 209247.8469, Evals: 2000

--- CMAME Step 5/200 ---
Archive: 476, Bins: 476, Coverage: 0.0476, Fitness min/mean/max: -102.33/-8.70/42.38, QD-score: 242142.0109, Evals: 2500

--- CMAME Step 6/200 ---
Archive: 548, Bins: 548, Coverage: 0.0548, Fitness min/mean/max: -102.33/-10.01/42.38, QD-score: 279487.0309, Evals: 3000

--- CMAME Step 7/200 ---
Archi